[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/18_montecarlo_search/notebooks/01_hex_y_rollouts.ipynb)

# Notebook 1: Hex y Rollouts

**Módulo 18 — Monte Carlo Tree Search · Notebook 01**

---

## ¿De qué trata este notebook?

En este notebook implementamos el juego de **Hex** desde cero y exploramos la idea de
evaluar posiciones mediante **rollouts** (partidas aleatorias). Es la base sobre la que
construiremos MCTS en los siguientes notebooks.

**Lo que vas a construir y explorar:**

1. La clase `Hex` con toda la lógica del juego.
2. Una función para mostrar el tablero como texto.
3. Partidas aleatorias completas y conteo de victorias.
4. Las funciones `ROLLOUT` y `EVALUAR_POR_ROLLOUTS` del pseudocódigo.
5. Ejercicio: estimar el valor de la celda central en Hex 3×3.
6. Gráfica de convergencia de rollouts.

**Relación con las notas de clase:** secciones 01 (Más allá de minimax) y 02 (Hex: el juego).

**Prerequisitos:** Python básico (clases), numpy, matplotlib.

---

In [ ]:
# Solo en Colab — en entorno local estas librerías ya deben estar instaladas
# !pip install numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import copy
from collections import deque

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "blue": "#2E86AB",
    "red": "#E94F37",
    "green": "#27AE60",
    "gray": "#7F8C8D",
    "orange": "#F39C12",
    "purple": "#8E44AD",
}

random.seed(42)
np.random.seed(42)

---

## 1. La clase `Hex`

Implementamos el juego de Hex siguiendo la misma interfaz que `TicTacToe` y `Nim`
del módulo 15: `actions()`, `result(a)`, `is_terminal()`, `utility(p)`.

**Convenciones:**
- Jugador 1 (Negro): conecta fila 0 (arriba) con fila $n-1$ (abajo)
- Jugador 2 (Blanco): conecta columna 0 (izquierda) con columna $n-1$ (derecha)
- Negro siempre empieza
- Los 6 vecinos hexagonales son: $(-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0)$

In [ ]:
class Hex:
    """Juego de Hex en un tablero de size x size.

    Jugador 1 (Negro): conecta arriba con abajo.
    Jugador 2 (Blanco): conecta izquierda con derecha.
    """

    # Direcciones de los 6 vecinos hexagonales
    NEIGHBORS = [(-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0)]

    def __init__(self, size=7):
        self.size = size
        # 0 = vacío, 1 = Negro, 2 = Blanco
        self.board = [[0] * size for _ in range(size)]
        self.current_player = 1  # Negro empieza

    def actions(self):
        """Retorna lista de celdas vacías (r, c)."""
        return [(r, c) for r in range(self.size)
                for c in range(self.size) if self.board[r][c] == 0]

    def result(self, action):
        """Aplica la acción y retorna un NUEVO estado."""
        new = copy.deepcopy(self)
        r, c = action
        new.board[r][c] = self.current_player
        new.current_player = 3 - self.current_player  # alterna 1 <-> 2
        return new

    def is_terminal(self):
        """True si algún jugador conectó sus lados."""
        return self._has_path(1) or self._has_path(2)

    def utility(self, player):
        """Retorna +1 si player ganó, -1 si perdió, 0 si no ha terminado."""
        if self._has_path(player):
            return 1
        if self._has_path(3 - player):
            return -1
        return 0

    def _has_path(self, player):
        """BFS para verificar si player conectó sus dos lados."""
        n = self.size
        if player == 1:  # Negro: fila 0 -> fila n-1
            start = [(0, c) for c in range(n) if self.board[0][c] == 1]
            goal = lambda r, c: r == n - 1
        else:  # Blanco: columna 0 -> columna n-1
            start = [(r, 0) for r in range(n) if self.board[r][0] == 2]
            goal = lambda r, c: c == n - 1

        visited = set()
        queue = deque(start)
        while queue:
            r, c = queue.popleft()
            if (r, c) in visited:
                continue
            visited.add((r, c))
            if goal(r, c):
                return True
            for dr, dc in self.NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and self.board[nr][nc] == player:
                    queue.append((nr, nc))
        return False

    def __repr__(self):
        return f"Hex(size={self.size}, turno={'Negro' if self.current_player == 1 else 'Blanco'})"

---

## 2. Visualización del tablero

Representamos el tablero como texto con desplazamiento para simular la geometría
hexagonal. Usamos `N` para Negro, `B` para Blanco, y `·` para vacío.

In [ ]:
def mostrar_tablero(game):
    """Muestra el tablero de Hex como texto con indentación hexagonal."""
    simbolos = {0: "·", 1: "N", 2: "B"}
    n = game.size
    # Encabezado con columnas
    header = "   " + " ".join(f"{c}" for c in range(n))
    print(header)
    for r in range(n):
        indent = " " * r  # desplazamiento hexagonal
        fila = " ".join(simbolos[game.board[r][c]] for c in range(n))
        print(f"{indent}{r:2d} {fila}")
    turno = "Negro (N)" if game.current_player == 1 else "Blanco (B)"
    print(f"\nTurno: {turno}")


# Probemos con un tablero vacío de 5x5
game = Hex(size=5)
mostrar_tablero(game)

In [ ]:
# Probemos jugando algunos movimientos manualmente
game = Hex(size=5)
game = game.result((2, 2))  # Negro en el centro
game = game.result((0, 0))  # Blanco en esquina
game = game.result((1, 2))  # Negro avanza
game = game.result((1, 1))  # Blanco bloquea

mostrar_tablero(game)
print(f"\nAcciones legales: {len(game.actions())} celdas vacías")
print(f"¿Terminal?: {game.is_terminal()}")

---

## 3. Partidas aleatorias

Antes de implementar rollouts formalmente, juguemos partidas completas al azar
y contemos quién gana. Esto nos da una primera intuición sobre el juego.

In [ ]:
def partida_aleatoria(size=5):
    """Juega una partida completa de Hex con movimientos aleatorios.

    Retorna (ganador, num_movimientos) donde ganador es 1 (Negro) o 2 (Blanco).
    """
    game = Hex(size=size)
    movimientos = 0
    while not game.is_terminal():
        action = random.choice(game.actions())
        game = game.result(action)
        movimientos += 1
    # Determinar quién ganó
    ganador = 1 if game.utility(1) == 1 else 2
    return ganador, movimientos


# Jugar 1000 partidas aleatorias en Hex 5x5
random.seed(42)
N_PARTIDAS = 1000
resultados = [partida_aleatoria(size=5) for _ in range(N_PARTIDAS)]

victorias_negro = sum(1 for g, _ in resultados if g == 1)
victorias_blanco = N_PARTIDAS - victorias_negro
duracion_media = np.mean([m for _, m in resultados])

print(f"Resultados de {N_PARTIDAS} partidas aleatorias en Hex 5×5:")
print(f"  Negro gana: {victorias_negro} ({100*victorias_negro/N_PARTIDAS:.1f}%)")
print(f"  Blanco gana: {victorias_blanco} ({100*victorias_blanco/N_PARTIDAS:.1f}%)")
print(f"  Duración media: {duracion_media:.1f} movimientos")
print(f"\nNota: Negro tiene ventaja de primer movimiento (teorema de Nash).")

---

## 4. Rollouts: evaluación por simulación

Un **rollout** es una partida aleatoria completa desde un estado dado hasta un
estado terminal. Es el estimador Monte Carlo del módulo 12 aplicado a juegos.

Implementamos dos funciones del pseudocódigo de las notas:

- `ROLLOUT(estado, jugador)`: una simulación aleatoria, retorna $+1$ o $-1$
- `EVALUAR_POR_ROLLOUTS(estado, jugador, N)`: promedia $N$ rollouts

In [ ]:
def rollout(state, player):
    """Juega al azar desde `state` hasta el final.

    Retorna la utilidad del estado terminal para `player`: +1 o -1.
    Corresponde a la función ROLLOUT del pseudocódigo [R1]-[R4].
    """
    sim = copy.deepcopy(state)              # [R1] copia el estado
    while not sim.is_terminal():            # [R2] juega hasta que termine
        action = random.choice(sim.actions())  # [R3] movimiento aleatorio
        sim = sim.result(action)
    return sim.utility(player)              # [R4] +1 o -1


def evaluar_por_rollouts(state, player, n_rollouts):
    """Estima el valor de `state` promediando n_rollouts simulaciones.

    Corresponde a EVALUAR_POR_ROLLOUTS del pseudocódigo [E1]-[E2].
    Retorna un valor en [-1, +1].
    """
    total = 0
    for _ in range(n_rollouts):             # [E1] acumular resultados
        total += rollout(state, player)
    return total / n_rollouts               # [E2] promedio = estimador MC

In [ ]:
# Evaluemos la posición inicial de Hex 3x3
random.seed(42)

state = Hex(size=3)
valor = evaluar_por_rollouts(state, player=1, n_rollouts=5000)

print(f"Valor estimado de la posición inicial (Hex 3×3, perspectiva Negro):")
print(f"  Estimador MC = {valor:.3f}")
print(f"  Interpretación: Negro gana ~{(1+valor)/2*100:.1f}% de las partidas aleatorias")
print(f"\nNota: por el teorema de Nash, Negro SIEMPRE tiene una estrategia ganadora.")
print(f"El valor positivo refleja la ventaja del primer movimiento.")

---

## 5. Evaluación de cada celda en Hex 3×3

¿Cuál es la mejor primera jugada para Negro en un tablero 3×3? Evaluemos cada
celda con rollouts: jugamos Negro en esa celda y medimos qué tan buena es la
posición resultante.

In [ ]:
random.seed(42)

size = 3
n_rollouts = 2000
state = Hex(size=size)

# Evaluar cada celda como primera jugada de Negro
valores = {}
for r in range(size):
    for c in range(size):
        # Negro juega en (r, c)
        siguiente = state.result((r, c))
        # Evaluamos desde la perspectiva de Negro (jugador 1)
        val = evaluar_por_rollouts(siguiente, player=1, n_rollouts=n_rollouts)
        valores[(r, c)] = val
        print(f"  Celda ({r},{c}): valor = {val:+.3f}")

# Mejor celda
mejor = max(valores, key=valores.get)
print(f"\nMejor primera jugada: celda {mejor} con valor {valores[mejor]:+.3f}")

In [ ]:
# Visualizar los valores como un mapa de calor
fig, ax = plt.subplots(figsize=(6, 5))

# Crear matriz de valores
val_matrix = np.zeros((size, size))
for (r, c), v in valores.items():
    val_matrix[r][c] = v

im = ax.imshow(val_matrix, cmap="RdYlGn", vmin=-1, vmax=1, aspect="equal")

# Anotar cada celda con su valor
for r in range(size):
    for c in range(size):
        v = val_matrix[r][c]
        color = "white" if abs(v) > 0.5 else "black"
        ax.text(c, r, f"{v:+.2f}", ha="center", va="center",
                fontsize=14, fontweight="bold", color=color)

ax.set_xticks(range(size))
ax.set_yticks(range(size))
ax.set_xlabel("Columna")
ax.set_ylabel("Fila")
ax.set_title(f"Valor estimado por rollouts — Primera jugada de Negro (Hex {size}×{size})")
plt.colorbar(im, ax=ax, label="Valor (perspectiva Negro)")
plt.tight_layout()
plt.show()

### Reflexión

Observa el mapa de calor:

1. **¿La celda central $(1,1)$ es la mejor?** En Hex, el centro suele ser
   estratégicamente fuerte porque tiene máxima conectividad.
2. **¿Las esquinas son buenas o malas?** Las esquinas tocan dos bordes, lo cual
   puede ser ventajoso.
3. **¿Los valores reflejan la simetría del tablero?** Con suficientes rollouts,
   las posiciones simétricas deberían tener valores similares.

---

## 6. Convergencia de los rollouts

Por la Ley de los Grandes Números (módulo 12), el estimador $\hat\mu_n$
converge al valor real conforme $n \to \infty$. Veamos la convergencia
gráficamente: calculamos la media acumulada de rollouts para tres celdas
del tablero 3×3.

In [ ]:
random.seed(42)

size = 3
N_TOTAL = 3000  # rollouts por celda
state = Hex(size=size)

# Tres celdas a comparar
celdas = [(1, 1), (0, 0), (0, 2)]
nombres = ["Centro (1,1)", "Esquina (0,0)", "Esquina (0,2)"]
colores = [COLORS["blue"], COLORS["red"], COLORS["green"]]

fig, ax = plt.subplots(figsize=(10, 6))

for celda, nombre, color in zip(celdas, nombres, colores):
    siguiente = state.result(celda)
    # Recopilar todos los rollouts
    resultados = np.array([rollout(siguiente, player=1) for _ in range(N_TOTAL)])
    # Media acumulada
    media_acum = np.cumsum(resultados) / np.arange(1, N_TOTAL + 1)
    ax.plot(media_acum, color=color, label=nombre, alpha=0.8)

ax.axhline(0, color=COLORS["gray"], linestyle="--", alpha=0.5)
ax.set_xlabel("Número de rollouts")
ax.set_ylabel("Valor estimado (media acumulada)")
ax.set_title("Convergencia del estimador por rollouts — Hex 3×3")
ax.legend()
ax.set_ylim(-0.5, 1.0)
plt.tight_layout()
plt.show()

print("Observa cómo la estimación se estabiliza después de ~500 rollouts.")
print("El error decrece como O(1/sqrt(n)) — independiente del tamaño del tablero.")

---

## 7. Ejercicio: estimar el valor de la celda central en distintos tamaños

**Tu tarea:** evalúa con rollouts el valor de jugar en la celda central para
tableros de distintos tamaños. ¿La ventaja del centro cambia con el tamaño?

Completa el código de abajo y responde:
1. ¿El valor del centro aumenta o disminuye con el tamaño del tablero?
2. ¿Por qué crees que ocurre eso?
3. ¿Cuántos rollouts necesitas para que la estimación sea estable en 5×5?

In [ ]:
random.seed(42)

sizes = [3, 5, 7]
n_rollouts = 2000  # <-- Experimenta con más o menos rollouts

print("Valor de la celda central por tamaño de tablero:\n")
for s in sizes:
    centro = (s // 2, s // 2)
    state = Hex(size=s)
    siguiente = state.result(centro)
    valor = evaluar_por_rollouts(siguiente, player=1, n_rollouts=n_rollouts)
    print(f"  Hex {s}×{s}, celda {centro}: valor = {valor:+.3f}")

print("\n¿Qué observas? Escribe tu respuesta aquí:")
# TU RESPUESTA:

---

## 8. Limitaciones de los rollouts puros

Los rollouts son un buen punto de partida, pero tienen dos problemas fundamentales:

1. **No reutilizan información**: si evaluamos 9 celdas con 2000 rollouts cada una,
   hicimos 18,000 simulaciones. Pero la información de un rollout para la celda
   $(0,0)$ no ayuda a evaluar la celda $(1,1)$.

2. **No enfocan el esfuerzo**: gastamos los mismos rollouts en celdas claramente
   malas que en las prometedoras. Un algoritmo inteligente debería dedicar más
   simulaciones a las celdas que parecen mejores.

Estos dos problemas motivan **MCTS** (notebook siguiente): un algoritmo que
construye un árbol para reutilizar información y concentra rollouts donde
más importan.

---